# 복소 벡터공간과 응용

> 선형대수 20강 · 응용 (마지막 강의)

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [복소 벡터공간과 응용](https://mioon1402.github.io/timeseriesdata/linalg/L20-applications.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 복소수 3분 요약

## 1. 왜 복소 고윳값이 나오는가

## 2. 복소 고윳값 = 회전 + 확대

## 3. 켤레전치와 에르미트 행렬

## 4. 유니터리 행렬과 푸리에 변환

## 5. PCA — 배운 것의 총집합

## 6. 페이지랭크 — 고유벡터로 검색 순위 매기기

## 7. 희소행렬 — 진짜 큰 문제

## 8. numpy 로 확인하기

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def 회전(도):
    t = np.radians(도)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

for 도 in (90, 30, 180):
    λ = np.linalg.eigvals(회전(도))
    크기 = np.abs(λ[0])
    각도 = np.degrees(np.angle(λ[0]))
    print(f"{도:3d}° 회전 → λ = {np.round(λ, 4)}"
          f"   |λ| = {크기:.4f}   arg λ = {abs(각도):.1f}°")
print("\n→ |λ| 는 확대율, arg λ 는 회전각. 고윳값이 변환을 그대로 설명한다.")

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

t = np.radians(32)
R = 0.95 * np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
B = np.array([[1.6, 0.], [0., 1.]])     # 가로로 늘려서 찌그러뜨린다
A = B @ R @ np.linalg.inv(B)

print("A =\n", A, "  ← 회전 행렬처럼 안 보인다")
λ = np.linalg.eigvals(A)
print("\nλ =", np.round(λ, 4))
print("|λ| =", round(float(np.abs(λ[0])), 4), " ← 0.95, 한 걸음마다 5% 줄어든다")
print("arg λ =", round(float(np.degrees(np.angle(λ[0]))), 2), "° ← 한 걸음마다 32도 회전")

x = np.array([3., 0.])
for k in (0, 10, 40):
    print(f"  A^{k:2d} x = {np.round(np.linalg.matrix_power(A, k) @ x, 4)}"
          f"   길이 {np.linalg.norm(np.linalg.matrix_power(A, k) @ x):.4f}")

In [ ]:
import numpy as np

z = np.array([1 + 1j, 1 - 1j])

print("z    =", z)
print("zᵀz  =", z @ z, "  ← 영벡터가 아닌데 0 이다!")
print("zᴴz  =", np.vdot(z, z), "  ← 켤레전치를 쓰면 제대로 나온다")
print("‖z‖  =", np.linalg.norm(z))
print()
print("np.vdot 은 첫 인자를 자동으로 켤레 취한다:",
      np.allclose(np.vdot(z, z), z.conj() @ z))

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

H = np.array([[2 + 0j, 1 - 1j],
              [1 + 1j, 3 + 0j]])

print("Hᴴ = H 인가:", np.allclose(H, H.conj().T))
λ, U = np.linalg.eigh(H)                # eigh 는 에르미트 전용
print("고윳값 =", λ, "  ← 전부 실수")
print("\n고유벡터가 직교하는가 (UᴴU = I):", np.allclose(U.conj().T @ U, np.eye(2)))
print("복원 UΛUᴴ = H:", np.allclose(U @ np.diag(λ) @ U.conj().T, H))
print("\n대각 성분은 반드시 실수:", H[0, 0], H[1, 1])

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

N = 4
j, k = np.meshgrid(np.arange(N), np.arange(N), indexing="ij")
F = np.exp(-2j * np.pi * j * k / N) / np.sqrt(N)

print("F =\n", np.round(F, 3))
print("\n유니터리인가 (FᴴF = I):", np.allclose(F.conj().T @ F, np.eye(N)))
print("고윳값의 크기 (전부 1):", np.round(np.abs(np.linalg.eigvals(F)), 4))

x = np.array([1., 2., 3., 4.])
print("\nF x       =", np.round(F @ x, 4))
print("np.fft/√N =", np.round(np.fft.fft(x) / np.sqrt(N), 4))
print("\n에너지 보존 (파스발):  ‖x‖ =", round(float(np.linalg.norm(x)), 6),
      "  ‖Fx‖ =", round(float(np.linalg.norm(F @ x)), 6))

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

rng = np.random.default_rng(3)
X = rng.normal(size=(200, 3)) @ np.array([[2., 1., 0.],
                                          [0., 1., 0.],
                                          [0., 0., 0.3]])
Xc = X - X.mean(axis=0)                  # ① 중심 이동

# 길 A — 공분산 행렬의 고유분해
C = Xc.T @ Xc / (len(X) - 1)             # ② 대칭행렬
λ = np.sort(np.linalg.eigvalsh(C))[::-1] # ③ 고유분해

# 길 B — X 의 SVD (실무에서 쓰는 길)
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
분산 = s ** 2 / (len(X) - 1)

print("공분산 고윳값 =", λ)
print("SVD 로 구한 분산 =", 분산)
print("같은가:", np.allclose(λ, 분산))
print()
print("설명 비율 =", np.round(분산 / 분산.sum() * 100, 2), "%")
print("→ 앞의 2개로", round(float((분산[:2].sum() / 분산.sum()) * 100), 1), "% 를 설명한다.")

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
U0, _ = np.linalg.qr(rng.normal(size=(50, 6)))
V0, _ = np.linalg.qr(rng.normal(size=(6, 6)))
X = U0 @ np.diag([1e3, 1e2, 10, 1, 0.1, 3.7e-3]) @ V0.T

cX = np.linalg.cond(X)
cC = np.linalg.cond(X.T @ X)
print(f"X 의 조건수      = {cX:.3e}")
print(f"XᵀX 의 조건수    = {cC:.3e}")
print(f"비율            = {cC / cX:.3e}   ← 대략 조건수만큼 나빠졌다 (제곱)")
print(f"\ndouble 정밀도 여유 ≈ 1e16 이므로 XᵀX 는 유효숫자를 크게 잃는다.")
print("→ PCA·최소제곱은 공분산을 만들지 말고 SVD/QR 을 직접 쓴다.")

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

이름 = ["A", "B", "C", "D"]
링크 = {0: [1, 2], 1: [2], 2: [0], 3: [2]}     # 페이지 j → 어디로

n = 4
A = np.zeros((n, n))
for j, outs in 링크.items():
    if outs:
        for i in outs:
            A[i, j] = 1 / len(outs)
    else:
        A[:, j] = 1 / n                        # 막다른 페이지는 균등 분배
d = 0.85
G = d * A + (1 - d) / n * np.ones((n, n))

r = np.ones(n) / n                             # ① 거듭제곱법
for k in range(1, 41):
    r = G @ r
    if k in (1, 3, 10, 40):
        print(f"  {k:2d}회 반복: {np.round(r, 4)}")

λ, V = np.linalg.eig(G)                        # ② 고유벡터로 직접
i = int(np.argmin(np.abs(λ - 1)))
직접 = np.real(V[:, i]); 직접 = 직접 / 직접.sum()
print("\nλ = 1 고유벡터:", np.round(직접, 4))
print("두 방법이 같은가:", np.allclose(r, 직접))
print("\n순위:", " > ".join(이름[i] for i in np.argsort(-r)))
print("두 번째 고윳값 크기 =", round(float(np.sort(np.abs(λ))[-2]), 4), " ← 감쇠계수 d 와 같다")

In [ ]:
import numpy as np
from scipy import sparse

n = 4000
rng = np.random.default_rng(1)
밀집 = np.zeros((n, n))
행 = rng.integers(0, n, 20000)
열 = rng.integers(0, n, 20000)
밀집[행, 열] = 1.0                      # 0.125% 만 채운다

희소 = sparse.csr_matrix(밀집)

print(f"밀집 행렬 : {밀집.nbytes / 1e6:8.2f} MB")
print(f"희소 행렬 : {(희소.data.nbytes + 희소.indices.nbytes + 희소.indptr.nbytes) / 1e6:8.2f} MB")
print(f"0 이 아닌 비율 : {희소.nnz / n**2 * 100:.3f}%")

x = rng.normal(size=n)
print("\n행렬 × 벡터 결과가 같은가:", np.allclose(밀집 @ x, 희소 @ x))
print("→ 거듭제곱법은 '행렬 × 벡터' 만 쓰므로 희소행렬로 그대로 돌아간다.")

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# 문제 1 — 0.6²+0.8²=1 이므로 순수 회전. |λ|=1 → 영원히 돌기만 한다
A1 = np.array([[0.6, -0.8], [0.8, 0.6]])
λ1 = np.linalg.eigvals(A1)
print("문제 1: λ =", np.round(λ1, 4), " |λ| =", round(float(np.abs(λ1[0])), 6))
print("        arg λ =", round(float(np.degrees(np.angle(λ1[0]))), 2), "°")
print("        → |λ|=1 인 순수 회전. 커지지도 작아지지도 않고 계속 돈다.")
A100 = np.linalg.matrix_power(A1, 100)
print("        A¹⁰⁰ =\n", A100)
print("        누적 회전각 =", round(float(np.degrees(np.arctan2(A100[1, 0], A100[0, 0])) % 360), 2), "°")

# 문제 2 — |det U| = 1 (복소수이므로 단위원 위의 아무 값)
N = 4
j, k = np.meshgrid(np.arange(N), np.arange(N), indexing="ij")
F = np.exp(-2j * np.pi * j * k / N) / np.sqrt(N)
print("\n문제 2: det F =", np.round(np.linalg.det(F), 4),
      "  |det F| =", round(float(np.abs(np.linalg.det(F))), 6))
print("        → 항상 |det| = 1. UᴴU=I 의 양변에 det 를 취하면 |det U|²=1 이 나온다.")
print("        실수 직교행렬이면 det = ±1 (회전 +1, 반사 −1 — 19강).")

# 문제 3 — d=1 이면 막다른 페이지·독점 무리 문제가 살아난다
링크 = {0: [1, 2], 1: [2], 2: [0], 3: [2]}
n = 4
A = np.zeros((n, n))
for jj, outs in 링크.items():
    for i in outs:
        A[i, jj] = 1 / len(outs)
r = np.ones(n) / n
for _ in range(200):
    r = A @ r
print("\n문제 3: d=1 일 때 r =", np.round(r, 4))
print("        → D 는 들어오는 링크가 없어 점수가 정확히 0 이 된다.")
print("        A·B·C 끼리만 점수를 돌려 D 는 영원히 소외된다.")
λA = np.linalg.eigvals(A)
print("        두 번째 고윳값 크기 =", round(float(np.sort(np.abs(λA))[-2]), 4),
      "← 1 에 가까워 수렴이 느려진다")
print("        감쇠(1−d)는 '아무 데나 점프' 를 넣어 이 두 문제를 한 번에 없앤다.")

## 9. 20강 전체를 한 장으로

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)